# Record two trials, save them and read them back

Experiment groups the conditions you tried and the data you obtained by Run. This Tutorial evaluates two candidates, saves their input, results and conditions in one file, and reads them in a new Python process or Notebook kernel.

This page is self-contained if you know Python and ordinary variables, objectives and constraints. Complete steps 1–2 of [Install the SDK](../getting_started/install.md), plus the kernel setup if using a Notebook. It uses `ommx==3.0.0b7`; no Adapter, solver or registry authentication is needed.

## 1. Prepare a small problem

Select at most one of two items. Rather than search for candidates, we evaluate two existing assignments. `evaluate` computes a candidate's objective value and feasibility, returning a `Solution`.

In [1]:
from ommx import Instance
from ommx.experiment import Experiment

instance = Instance.maximize()
x = instance.new_binary("x")
y = instance.new_binary("y")
instance.objective = 3 * x + 2 * y
_ = instance.add_constraint(x + y <= 1)

## 2. Record conditions and results in Runs

An Experiment groups a series of trials, and a Run represents one trial within it. Attach the shared problem to the Experiment and each candidate's evaluation result to its Run. Use `log_parameter` to explicitly record scalar values you want to list later.

In [2]:
with Experiment() as experiment:
    experiment.log_instance("problem", instance)

    for label, candidate in [
        ("choose-x", {x.id: 1, y.id: 0}),
        ("choose-y", {x.id: 0, y.id: 1}),
    ]:
        with experiment.run() as run:
            solution = instance.evaluate(candidate)
            run.log_parameter("candidate", label)
            run.log_parameter("objective", solution.objective)
            run.log_parameter("feasible", solution.feasible)
            run.log_solution("evaluation", solution)

Exiting each `with` block closes its Run, and finally commits the Experiment. Records are written to the Local Registry, OMMX's local storage. You do not need to manipulate its layout or names here.

## 3. Read the recorded conditions

In [3]:
experiment.run_parameters_df()

,candidate,feasible,objective
run_id,,,
0,choose-x,True,3.0
1,choose-y,True,2.0


Both candidates are feasible. `choose-x` has objective value 3 and `choose-y` has value 2. These columns contain the values explicitly recorded by `log_parameter`; Experiment does not automatically capture every Python variable or the whole execution environment.

The saved `Solution` in each Run retains candidate variable values as well as its objective. Saving the problem and evaluation results lets you retrieve the data later, rather than merely display a table. These are candidate evaluations, not records of solver-proven optimality.

## 4. Save a file

In [4]:
experiment.save("candidate-trials.ommx")

The file contains the shared problem, both Runs' conditions and each evaluation result. You can now keep this file and end the Python process that created it. Saving does not overwrite an existing file; when rerunning, choose another output filename and update the later import path too.

## 5. Open it in a new process or kernel

For a Notebook, restart the kernel and run from the following cell. For a script, run the same code in another Python process. The current directory must contain `candidate-trials.ommx`; use its actual path if you moved it.

In [5]:
from ommx.experiment import Experiment

loaded = Experiment.import_archive("candidate-trials.ommx")
loaded.run_parameters_df()

,candidate,feasible,objective
run_id,,,
0,choose-x,True,3.0
1,choose-y,True,2.0


`import_archive` imports the file into the reader's Local Registry. It does not need the original `experiment` or `instance` Python objects.

In [6]:
problem = loaded.get_instance("problem")
for run in loaded.runs:
    saved = run.get_solution("evaluation")
    checked = problem.evaluate(saved.state)
    assert checked.objective == saved.objective
    assert checked.feasible == saved.feasible
    print(run.run_id, run.status, saved.objective, saved.feasible)

0 finished 3.0 True
1 finished 2.0 True


The saved Runs, input and results are now available again. A `status` of `finished` means that the Run's scope completed normally; it does not establish optimality or feasibility.

You have completed recording trial conditions and data, saving them, and retrieving them from a file. Reading saved results is different from rerunning a computation in the same environment. Explicitly retain the inputs, code, settings and environment information needed for that separate purpose.

## When needed

- [Experiment records and lifecycle](../user_guide/experiment.md): Runs, solver calls and the scope of automatic recording.
- [Continue or restore an Experiment](../how_to/continue_experiment.md): fork committed records or recover from checkpoints.
- [Attach data to an Experiment](../how_to/attach_experiment_data.md): JSON, files and typed data.
- [Exchange data with an Artifact](./share_in_ommx_artifact): save and exchange problems or results without a trial hierarchy.